# South Azerbaijani ASR — Colab inference workshop

This hands-on tutorial runs a Kartal Ol Whisper checkpoint directly from [Hugging Face](https://huggingface.co/Kartal-Ol/ASR-AZB), transcribes one audio sample, and analyzes WER, CER, DIR, and sample-level SER. It works on CPU, but a Colab GPU is recommended (**Runtime → Change runtime type → T4 GPU**).

> **Metric terminology:** WER and CER are percentages in the paper, and DIR is the raw deletion/insertion ratio. SER is included here only as an intuitive workshop diagnostic: for one utterance it is either 0% or 100%. It does not replace DIR in the official benchmark.

## 1. Install the lightweight demo dependencies

In [ ]:
%pip install -q "transformers>=4.46" "datasets>=3.0" "accelerate>=1.0" "librosa>=0.10.2" "soundfile>=0.12" "jiwer>=3.0"

## 2. Select a public checkpoint

Change only `MODEL_SUBFOLDER` to try another checkpoint. The full-dataset Base model is kept separately from the Community-only Base model.

In [ ]:
MODEL_REPO = "Kartal-Ol/ASR-AZB"
MODEL_SUBFOLDER = "whisper-base-full"  #@param ["whisper-tiny", "whisper-base", "whisper-base-full", "whisper-Small", "whisper-Small-Farsi", "whisper-small-north-azerbaijani", "whisper-small-turkish", "whisper-Small-Arabic"]
MAX_NEW_TOKENS = 225  #@param {type:"integer"}

print(f"Checkpoint: https://huggingface.co/{MODEL_REPO}/tree/main/{MODEL_SUBFOLDER}")

## 3. Load the model and processor from Hugging Face

In [ ]:
import torch
from transformers import AutoProcessor, WhisperForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_REPO, subfolder=MODEL_SUBFOLDER)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_REPO, subfolder=MODEL_SUBFOLDER, torch_dtype=dtype, low_cpu_mem_usage=True
).to(device)
model.eval()
print(f"Loaded {MODEL_SUBFOLDER} on {device}.")

## 4. Upload and preview one sample

Accepted formats include WAV, FLAC, MP3, and OGG when the Colab audio backend can decode them. The waveform is automatically converted to mono and resampled to 16 kHz.

In [ ]:
from google.colab import files
from IPython.display import Audio, display
import librosa

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No audio file was uploaded.")
audio_path = next(iter(uploaded))
waveform, sampling_rate = librosa.load(audio_path, sr=16_000, mono=True)
print(f"{audio_path}: {len(waveform) / sampling_rate:.2f} seconds at {sampling_rate} Hz")
display(Audio(waveform, rate=sampling_rate))

## 5. Transcribe

In [ ]:
inputs = processor(waveform, sampling_rate=sampling_rate, return_tensors="pt")
input_features = inputs.input_features.to(device=device, dtype=dtype)
with torch.inference_mode():
    predicted_ids = model.generate(input_features, max_new_tokens=MAX_NEW_TOKENS)
prediction = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0].strip()
print("Prediction:")
print(prediction)

## 6. One-sample error analysis

Paste the exact human reference below. By default, scoring only performs Unicode NFC and whitespace normalization. This is intentionally conservative and does **not** silently apply the paper's South Azerbaijani benchmark normalizer.

In [ ]:
REFERENCE = ""  #@param {type:"string"}

import math
import re
import unicodedata
import jiwer
import pandas as pd

def conservative_normalize(text: str) -> str:
    return re.sub(r"\s+", " ", unicodedata.normalize("NFC", text)).strip()

reference_scored = conservative_normalize(REFERENCE)
prediction_scored = conservative_normalize(prediction)
if not reference_scored:
    raise ValueError("Enter a non-empty REFERENCE, then run this cell again.")

word_result = jiwer.process_words(reference_scored, prediction_scored)
wer = 100.0 * word_result.wer
cer = 100.0 * jiwer.cer(reference_scored, prediction_scored)
dir_value = (word_result.deletions / word_result.insertions
             if word_result.insertions else (math.inf if word_result.deletions else 0.0))
ser = 0.0 if reference_scored == prediction_scored else 100.0

metrics = {
    "WER (%)": wer, "CER (%)": cer, "DIR (raw D/I)": dir_value,
    "SER (%)": ser, "Substitutions": word_result.substitutions,
    "Deletions": word_result.deletions, "Insertions": word_result.insertions,
}
display(pd.DataFrame([metrics]).round(3))
print("Reference :", reference_scored)
print("Prediction:", prediction_scored)
print("\nWord alignment:")
print(jiwer.visualize_alignment(word_result))

### How to read the metrics

- **WER (%)**: word substitutions + deletions + insertions, divided by reference words.
- **CER (%)**: the analogous character error percentage.
- **DIR**: word deletions divided by insertions; it is a raw ratio, not a percentage. If there are deletions but no insertions, it is infinity.
- **SER (%)**: percentage of utterances containing at least one error. With exactly one sample it can only be 0% or 100%, so use WER/CER for a more informative diagnosis.

## 7. Inspect the public GoldSet metadata (optional)

The public Hugging Face GoldSet table currently stores references and relative audio paths. It does not package the audio blobs in the dataset repository, so this cell inspects metadata only; use an authorized local audio copy with the upload cell above for inference.

In [ ]:
from datasets import load_dataset

goldset = load_dataset("Kartal-Ol/AZB-ASR-Gold-Testset", split="train")
print(goldset)
display(goldset.select(range(min(5, len(goldset)))).to_pandas())

## 8. Save the result

In [ ]:
import json

result = {
    "model": MODEL_REPO, "subfolder": MODEL_SUBFOLDER,
    "audio": audio_path, "reference": REFERENCE,
    "prediction": prediction, "metrics": metrics,
}
with open("azb_asr_demo_result.json", "w", encoding="utf-8") as handle:
    json.dump(result, handle, ensure_ascii=False, indent=2, allow_nan=True)
files.download("azb_asr_demo_result.json")

## Next steps

For reproducible corpus-level evaluation with the official repository normalization and JSON/CSV outputs, use [`scripts/evaluate.py`](../scripts/evaluate.py). See the project [README](../README.md) for training and benchmark commands.